# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Open Book, Eigener Computer, Internet-Zugang</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Nicht erlaubt ist der systematische Einsatz von generativer KI (z.B. Copilot, ChatGPT, Claude, Google's «Übersicht mit KI») für das Lösen der Aufgaben sowie beliebige Formen von Kommunikation oder Kollaboration mit anderen Menschen.</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (cinema.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (cinema.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Antwort bzw. Ansatz im Kern falsch (z.B. falsches Konzept oder falsche Schlussfolgerung; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `cinema.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `movies`, `cinemas`, `staff`, `screenings`, `customers`, `tickets`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `cinema_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [62]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /workspaces/python_postgresql_pgadmin/AP01_V2


### Datenbankverbindung zu PostgreSQL

In [63]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

Host:       localhost
Datenbank:  postgres


### Daten nach PostgreSQL laden

In [64]:
# Tabellen aus der SQLite-Datei cinema.db nach PostgreSQL übertragen
tables = ['movies', 'cinemas', 'staff', 'screenings', 'customers', 'tickets']

sqlite_engine = create_engine('sqlite:///cinema.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

movies         15 Zeilen
cinemas         6 Zeilen
staff          30 Zeilen
screenings    300 Zeilen
customers     505 Zeilen
tickets      2000 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `cinema.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///cinema.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `cinema.db` werden nicht gewertet.*

### A1 (4 Punkte)

Die Tabelle `screenings` enthält `base_price` und die Tabelle `tickets` enthält `paid_price`.

Ein Datenbankarchitekt sagt: *«`paid_price` ist redundant — der Preis steht bereits in `screenings.base_price`. Die Spalte sollte entfernt werden.»*

Hat er recht? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Kinobetrieb (z.B. Rabattaktionen, Gutscheine).

**Ihre Antwort:**

> Nein, der Datenbankarchitekt hat nicht recht.
>
>base_price und paid_price speichern unterschiedliche Informationen:
>
> * screenings.base_price ist der reguläre Basispreis für eine Vorstellung.
> * tickets.paid_price ist der Preis, der für ein konkretes Ticket tatsächlich bezahlt wurde.
>
> Beispiel: Der Basispreis einer Vorstellung beträgt CHF 20.–. Ein Kunde erhält aufgrund einer Rabattaktion 25 % Rabatt und bezahlt nur CHF 15.–. Dann gilt:
>
> base_price = 20.00
> paid_price = 15.00
>
> paid_price ist deshalb nicht redundant, da der tatsächlich bezahlte Preis nicht immer aus base_price abgeleitet werden kann. Zudem bleibt so der historische Verkaufspreis eines Tickets gespeichert, auch wenn sich 
>Preise oder Rabattregeln später ändern.

### A2 (4 Punkte)

Das Datenteam analysiert die Tabelle `customers` und findet folgende Werte in der Spalte `membership_type`:

```
Standard, standard, STANDARD, Standart, Premium, premium, Student, student, Senior
```

Beschreiben Sie das konkrete Datenproblem und nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (z.B. accurate, consistent, complete, reliable).

**Ihre Antwort:**

> Die Werte in membership_type sind nicht konsistent, da derselbe Mitgliedschaftstyp unterschiedlich geschrieben wird, z. B. Standard, standard und STANDARD. Zusätzlich enthält Standart einen Tippfehler. Die betroffene 
> Data-Quality-Eigenschaft ist daher Consistency (consistent).

## Data Quality – Eigenschaften

| Eigenschaft | Bedeutung | Beispiel `customers` | Typisches Signal |
|---|---|---|---|
| **Accurate** | Daten sind **inhaltlich korrekt** und entsprechen der Realität. | Kunde ist tatsächlich `Student`, aber in der DB steht `Senior`. Oder `Standart` statt des vorgesehenen Werts `Standard`. | **Ist der Wert richtig?** |
| **Consistent** | Gleiche Informationen werden **einheitlich dargestellt** bzw. widersprechen sich nicht. | `Standard`, `standard`, `STANDARD` für dieselbe Kategorie. | **Ist es überall einheitlich?** |
| **Complete** | Benötigte Daten sind **vorhanden**; es fehlen keine relevanten Werte. | `membership_type` ist bei einigen Kunden `NULL` oder leer. | **Fehlt etwas?** |
| **Reliable** | Auf die Daten bzw. ihre Quelle kann man sich **verlässlich verlassen**. | Mitgliedschaftsdaten stammen teilweise aus einem ungeprüften Import und es ist unklar, ob sie aktuell bzw. vertrauenswürdig sind. | **Kann ich den Daten vertrauen?** |

### Merkhilfe

- **Accurate** → Ist der Wert **richtig**?
- **Consistent** → Ist es überall **einheitlich**?
- **Complete** → **Fehlt** etwas?
- **Reliable** → Kann ich den Daten **vertrauen**?

### A3 (4 Punkte)

In `cinema.db` ist `paid_price` bei keinem Ticket höher als der `base_price` der zugehörigen Vorstellung. Angenommen, ein neues Ticket hätte `paid_price` = CHF 18.00 bei einem `base_price` von CHF 16.00: Wäre das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**

> Nein, nicht zwingend. Ein paid_price von CHF 18.00 bei einem base_price von CHF 16.00 könnte beispielsweise durch einen Zuschlag (z. B. für 3D, Premium-Sitzplatz oder eine Buchungsgebühr) entstehen und wäre daher nicht > automatisch ein Datenfehler.

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `cinema.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl verkaufter Tickets pro Kino — für **alle** Kinos in der Datenbank.

```sql
SELECT c.name, COUNT(*) AS ticket_count
FROM screenings s
JOIN tickets t ON s.screeningid = t.screeningid
JOIN cinemas c ON s.cinemaid = c.cinemaid
GROUP BY c.name;
```

Die Tabelle `cinemas` enthält **6 Kinos**. Tatsächliches Ergebnis der SQL-Abfrage: nur **5 Zeilen**, Ticketanzahl zwischen 350 und 450.

Das Management fragt: *«Zeig mir auch Kinos, die noch keine Tickets verkauft haben.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 6 Kinos** erscheinen und ein Kino ohne Tickets korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> cinemas wird als Ausgangstabelle verwendet, damit alle Kinos im Ergebnis berücksichtigt werden. Die Verknüpfungen werden mit LEFT JOIN durchgeführt, damit auch Kinos ohne zugehörige Screenings oder Tickets erhalten bleiben. COUNT(t.ticketid) wird anstelle von COUNT(*) verwendet, weil COUNT(t.ticketid) die NULL-Werte ignoriert und ein Kino ohne verkaufte Tickets dadurch korrekt mit 0 ausgewiesen wird.

**Korrigierte SQL-Abfrage:**

In [71]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
SELECT c.name, COUNT(t.ticketid) AS ticket_count

FROM cinemas c

LEFT JOIN screenings s ON s.cinemaid = c.cinemaid

LEFT JOIN tickets t ON s.screeningid = t.screeningid

GROUP BY c.name
ORDER BY ticket_count DESC;
""", con=engine)

engine.dispose()
df_B1

,name,ticket_count
0,Kinok,441
1,Pathé Küchlin,436
2,Rex Kino,386
3,Cineplexx,383
4,Arthouse Metropol,354
5,Kino Aurora,0


### B2 (8 Punkte)

**Ziel:** Liste aller Kunden (`customerid`, `firstname`, `lastname`), die noch nie ein Ticket gekauft haben.

```sql
SELECT c.customerid, c.firstname, c.lastname
FROM customers c
JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
```

Tatsächliches Ergebnis: 0 Zeilen — obwohl es Kunden ohne Tickets gibt.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, die alle Kunden ohne Ticket ausgibt. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> Der normale JOIN (INNER JOIN) wird durch einen LEFT JOIN ersetzt. Ein INNER JOIN gibt nur Kunden zurück, für die ein passendes Ticket existiert. Dadurch können Kunden ohne Tickets gar nicht im Ergebnis vorkommen. Der LEFT JOIN behält hingegen alle Kunden und setzt die Spalten von tickets auf NULL, wenn kein passendes Ticket existiert. Mit WHERE t.ticketid IS NULL werden anschließend genau diese Kunden ohne Ticket ausgewählt.

**Korrigierte SQL-Abfrage:**

In [72]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT c.customerid, c.firstname, c.lastname
FROM customers c
LEFT JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL
ORDER BY customerid;

""", con=engine)

engine.dispose()
df_B2

,customerid,firstname,lastname
0,CU0019,David,Krüger
1,CU0045,Sara,Schmidt
2,CU0165,Liam,Richter
3,CU0259,Max,Keller
4,CU0270,Marie,Braun
5,CU0334,Mia,Krüger
6,CU0360,Tom,Wolf
7,CU0432,Lucas,Wolf
8,CU0501,Anna,Müller
9,CU0502,Thomas,Berger


---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `cinema.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `customers.membership_type` sind `Standard`, `Premium`, `Student` und `Senior`. Schreiben Sie eine SQL-Abfrage, die alle Kunden (`customerid`, `firstname`, `lastname`, `membership_type`) mit einem ungültigen Wert in `membership_type` ausgibt — d.h. alle Werte ausser diesen vier. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

> NOT IN filtert alle Kunden, deren membership_type keinem der vier gültigen Werte (Standard, Premium, Student, Senior) entspricht. Somit werden nur Kunden mit einem ungültigen membership_type ausgegeben.

**SQL-Abfrage:**

In [73]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT customerid, firstname, lastname, membership_type FROM Customers
WHERE membership_type NOT IN ('Standard', 'Premium', 'Student', 'Senior')
ORDER BY customerid

""", con=engine)

engine.dispose()
df_C1

,customerid,firstname,lastname,membership_type
0,CU0501,Anna,Müller,standard
1,CU0502,Thomas,Berger,STANDARD
2,CU0503,Julia,Schmid,Standart
3,CU0504,Kevin,Keller,premium
4,CU0505,Sandra,Fischer,student


### C2 (4 Punkte)

Wie hoch ist der durchschnittliche bezahlte Ticketpreis (`paid_price`) pro Mitgliedschaftstyp (`membership_type`)? Geben Sie `membership_type` und den Durchschnittspreis (`avg_price`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_price`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> Die Tabellen customers und tickets müssen über customerid verbunden werden, da membership_type in customers und paid_price in tickets gespeichert ist. Anschließend wird nach membership_type gruppiert, um mit AVG(t.paid_price) den durchschnittlich bezahlten Ticketpreis pro Mitgliedschaftstyp zu berechnen.

**SQL-Abfrage:**

In [68]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT c.membership_type,  ROUND(CAST(AVG(t.paid_price) AS NUMERIC), 2) AS avg_price FROM customers AS c
INNER JOIN Tickets AS t
ON c.customerid = t.customerid
GROUP BY c.membership_type
ORDER BY avg_price DESC;

""", con=engine)

engine.dispose()
df_C2

,membership_type,avg_price
0,Premium,15.62
1,Student,15.59
2,Standard,15.49
3,Senior,15.34


### C3 (4 Punkte)

Welche Kinos haben mehr als 400 Tickets verkauft? Geben Sie `name` und Anzahl Tickets (`ticket_count`) aus, absteigend sortiert nach `ticket_count`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> Die Tabellen cinemas, screenings und tickets müssen verbunden werden, da ein Ticket über screeningid einer Vorstellung und diese über cinemaid einem Kino zugeordnet ist. Nach der Gruppierung pro Kino werden mit HAVING COUNT(*) > 400 nur Kinos ausgewählt, die mehr als 400 Tickets verkauft haben.

**SQL-Abfrage:**

In [69]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT c.name, COUNT(*) AS ticket_count
FROM screenings s
INNER JOIN tickets t ON s.screeningid = t.screeningid
INNER JOIN cinemas c ON s.cinemaid = c.cinemaid
GROUP BY c.name
HAVING COUNT(*) > 400
ORDER BY ticket_count DESC;

""", con=engine)

engine.dispose()
df_C3

,name,ticket_count
0,Kinok,441
1,Pathé Küchlin,436


# 📘 SQL – Prüfungsvorlagen & Begründungen

> **Ziel:** Schnell erkennen, welche SQL-Clause benötigt wird, und direkt eine passende Begründung formulieren können.

---

## 1. `SELECT`

**Wann?** Wenn bestimmte Spalten ausgegeben werden sollen.

```sql
SELECT firstname, lastname
FROM customers;
```

**Begründung:**  
`SELECT` wird verwendet, um die für die Aufgabenstellung benötigten Spalten `[Spalten]` auszugeben.

> **Merke:** `SELECT` = Was möchte ich sehen?

---

## 2. `SELECT DISTINCT`

**Wann?** Wenn jeder unterschiedliche Wert nur einmal erscheinen soll.

```sql
SELECT DISTINCT membership_type
FROM customers;
```

**Begründung:**  
`DISTINCT` entfernt Duplikate, sodass jeder unterschiedliche Wert nur einmal im Ergebnis erscheint.

> **Merke:** `DISTINCT` = keine Duplikate.

---

## 3. `FROM`

**Wann?** Legt die Ausgangstabelle der Abfrage fest.

```sql
SELECT firstname
FROM customers;
```

**Begründung:**  
Die benötigten Daten befinden sich in `[Tabelle]`, weshalb diese als Ausgangstabelle verwendet wird.

**Bei `LEFT JOIN`:**  
`[Tabelle X]` wird als Ausgangstabelle verwendet, da alle Datensätze dieser Tabelle im Ergebnis berücksichtigt werden sollen.

> **Merke:** `FROM` = Woher kommen meine Daten?

---

## 4. `WHERE`

**Wann?** Wenn einzelne Datensätze anhand einer Bedingung gefiltert werden sollen.

```sql
WHERE paid_price > 15
```

**Begründung:**  
`WHERE` filtert die Datensätze und behält nur diejenigen, welche die Bedingung `[Bedingung]` erfüllen.

> **Merke:** `WHERE` filtert einzelne Zeilen **vor** einer Gruppierung.

---

## 5. `AND`

**Wann?** Wenn mehrere Bedingungen gleichzeitig erfüllt sein müssen.

```sql
WHERE membership_type = 'Student'
AND paid_price < 15
```

**Begründung:**  
`AND` verknüpft mehrere Bedingungen, sodass ein Datensatz alle angegebenen Bedingungen gleichzeitig erfüllen muss.

> **Merke:** `AND` = Bedingung 1 **UND** Bedingung 2.

---

## 6. `LIKE`

**Wann?** Wenn Text anhand eines Musters gesucht werden soll.

```sql
WHERE lastname LIKE 'M%'
```

| Ausdruck | Bedeutung |
|---|---|
| `'M%'` | beginnt mit M |
| `'%M'` | endet mit M |
| `'%M%'` | enthält M |

**Begründung:**  
`LIKE` wird verwendet, um Textwerte anhand des angegebenen Musters zu filtern.

> **Merke:** `%` steht für beliebig viele Zeichen.

---

## 7. `GROUP BY`

**Wann?** Wenn etwas **pro Gruppe/Kategorie** berechnet werden soll.

Typische Formulierungen:
- Anzahl Tickets **pro Kino**
- Durchschnittspreis **pro Mitgliedschaftstyp**
- Umsatz **pro Kunde**

```sql
SELECT membership_type, AVG(paid_price)
FROM ...
GROUP BY membership_type;
```

**Begründung:**  
`GROUP BY [Spalte]` fasst die Datensätze nach `[Spalte]` zusammen, damit die Aggregatfunktion `[COUNT / AVG / SUM / ...]` für jede Gruppe separat berechnet werden kann.

> **Merke:** „pro ...“ → meistens `GROUP BY`.

---

## 8. `HAVING`

**Wann?** Wenn nach einem **aggregierten Ergebnis** gefiltert wird.

```sql
GROUP BY c.name
HAVING COUNT(t.ticketid) > 400
```

**Begründung:**  
`HAVING` wird verwendet, um die durch `GROUP BY` gebildeten Gruppen anhand eines aggregierten Werts zu filtern.

| Clause | Filtert |
|---|---|
| `WHERE` | einzelne Zeilen **vor** `GROUP BY` |
| `HAVING` | Gruppen **nach** `GROUP BY` |

> **Merke:** Filter auf `COUNT`, `AVG`, `SUM`, ... → `HAVING`.

---

## 9. `ORDER BY`

**Wann?** Wenn das Ergebnis sortiert werden soll.

```sql
ORDER BY avg_price DESC;
```

| Befehl | Bedeutung |
|---|---|
| `ASC` | aufsteigend |
| `DESC` | absteigend |

**Begründung:**  
`ORDER BY [Spalte] DESC` sortiert das Ergebnis absteigend nach `[Spalte]`.

> **Merke:** `ASC` = klein → gross, `DESC` = gross → klein.

---

# 🔗 JOINs

## 10. `FULL JOIN`

**Wann?** Wenn alle Datensätze **beider Tabellen** erhalten bleiben sollen.

```sql
FROM table_a a
FULL JOIN table_b b
    ON a.id = b.id
```

**Begründung:**  
`FULL JOIN` behält alle Datensätze aus beiden Tabellen, unabhängig davon, ob auf der jeweils anderen Seite eine Übereinstimmung existiert.

> **Merke:** `FULL JOIN` = alles von beiden Seiten.

---

## 11. `INNER JOIN`

**Wann?** Wenn nur Datensätze benötigt werden, für die in **beiden Tabellen eine Übereinstimmung** existiert.

```sql
FROM customers c
INNER JOIN tickets t
    ON c.customerid = t.customerid
```

**Begründung:**  
`INNER JOIN` wird verwendet, da nur Datensätze benötigt werden, für die in beiden Tabellen eine Übereinstimmung existiert.

**Vorlage bei Informationen aus zwei Tabellen:**  
`[Information A]` liegt in `[Tabelle A]` und `[Information B]` in `[Tabelle B]`. Deshalb werden die Tabellen über `[gemeinsame ID]` verbunden.

**Beispiel:**  
`membership_type` liegt in `customers` und `paid_price` in `tickets`. Deshalb werden beide Tabellen über `customerid` verbunden.

> **Merke:** `INNER JOIN` = nur Matches.

---

## 12. `LEFT JOIN`

**Wann?** Wenn **alle Datensätze der linken Tabelle** erscheinen sollen, auch wenn rechts nichts Passendes existiert.

```sql
FROM customers c
LEFT JOIN tickets t
    ON c.customerid = t.customerid
```

**Begründung:**  
`LEFT JOIN` wird verwendet, damit alle Datensätze aus `[linke Tabelle]` erhalten bleiben, auch wenn kein zugehöriger Datensatz in `[rechte Tabelle]` existiert.

Existiert rechts kein passender Datensatz, entstehen dort `NULL`-Werte.

> **Merke:** „Alle X, auch ohne Y“ → `X LEFT JOIN Y`.

---

## 13. `RIGHT JOIN`

**Wann?** Wenn alle Datensätze der **rechten Tabelle** erhalten bleiben sollen.

```sql
FROM tickets t
RIGHT JOIN customers c
    ON t.customerid = c.customerid
```

**Begründung:**  
`RIGHT JOIN` wird verwendet, damit alle Datensätze der rechten Tabelle erhalten bleiben, auch wenn in der linken Tabelle keine Übereinstimmung existiert.

> **Merke:** Wie `LEFT JOIN`, nur umgekehrt.

---

## 14. `ON`

**Wann?** Definiert, über welche Spalten zwei Tabellen verbunden werden.

```sql
ON c.customerid = t.customerid
```

**Begründung:**  
`ON` definiert die Verknüpfungsbedingung und verbindet die zusammengehörigen Datensätze über `[ID]`.

**Beispiel:**  
`ON c.customerid = t.customerid` ordnet jedes Ticket dem zugehörigen Kunden zu.

---

## 15. `LIMIT`

**Wann?** Wenn nur eine bestimmte Anzahl Ergebnisse ausgegeben werden soll.

```sql
ORDER BY paid_price DESC
LIMIT 5;
```

**Begründung:**  
`LIMIT 5` beschränkt die Ausgabe auf maximal fünf Datensätze.

> **Merke:** `ORDER BY ... DESC` + `LIMIT 5` → z.B. die 5 höchsten Werte.

---

## 16. `AS`

**Wann?** Um einer Tabelle oder berechneten Spalte einen Alias zu geben.

```sql
AVG(paid_price) AS avg_price
```

oder:

```sql
FROM customers AS c
```

**Begründung:**  
`AS` vergibt einen kürzeren bzw. aussagekräftigeren Alias für eine Spalte oder Tabelle.

---

# 🧮 Aggregatfunktionen

## 17. `AVG`

**Wann?** Wenn der Durchschnitt berechnet werden soll.

```sql
AVG(t.paid_price)
```

**Begründung:**  
`AVG(t.paid_price)` berechnet den durchschnittlich bezahlten Ticketpreis.

> **Merke:** `AVG` = Durchschnitt.

---

## 18. `COUNT`

**Wann?** Wenn die Anzahl bestimmt werden soll.

```sql
COUNT(t.ticketid)
```

**Begründung:**  
`COUNT(t.ticketid)` zählt alle vorhandenen Nicht-`NULL`-Werte von `ticketid` und damit die entsprechenden Tickets.

### `COUNT(*)` vs. `COUNT(Spalte)`

```sql
COUNT(*)
```

→ zählt **alle Ergebniszeilen**.

```sql
COUNT(t.ticketid)
```

→ zählt nur **Nicht-NULL-Werte**.

**Besonders wichtig bei `LEFT JOIN`:**  
`COUNT(t.ticketid)` wird verwendet, da `NULL`-Werte nicht mitgezählt werden und ein Datensatz ohne zugehöriges Ticket dadurch korrekt den Wert `0` erhält.

> **Merke:** Bei `LEFT JOIN` + „Anzahl inklusive 0“ → meistens `COUNT(rechte_tabelle.id)`.

---

## 19. `MIN`

**Wann?** Wenn der kleinste Wert bestimmt werden soll.

```sql
MIN(paid_price)
```

**Begründung:**  
`MIN(paid_price)` bestimmt den kleinsten bezahlten Ticketpreis.

---

## 20. `MAX`

**Wann?** Wenn der grösste Wert bestimmt werden soll.

```sql
MAX(paid_price)
```

**Begründung:**  
`MAX(paid_price)` bestimmt den grössten bezahlten Ticketpreis.

---

## 21. `SUM`

**Wann?** Wenn Werte addiert werden sollen.

```sql
SUM(paid_price)
```

**Begründung:**  
`SUM(paid_price)` addiert die Werte und berechnet damit die Summe der bezahlten Ticketpreise.

---

# 🔎 Filterbedingungen

## 22. `BETWEEN`

**Wann?** Wenn Werte innerhalb eines Bereichs gesucht werden.

```sql
WHERE paid_price BETWEEN 10 AND 20
```

**Begründung:**  
`BETWEEN 10 AND 20` wählt alle Werte innerhalb des angegebenen Bereichs inklusive der beiden Grenzwerte aus.

> **Merke:** `BETWEEN` schliesst beide Grenzen ein.

---

## 23. `IN`

**Wann?** Wenn ein Wert einem von mehreren vorgegebenen Werten entsprechen soll.

```sql
WHERE membership_type IN ('Student', 'Senior')
```

**Begründung:**  
`IN` wird verwendet, um nur Datensätze auszuwählen, deren `membership_type` einem der angegebenen Werte entspricht.

### `NOT IN`

```sql
WHERE membership_type NOT IN
('Standard', 'Premium', 'Student', 'Senior')
```

**Begründung:**  
`NOT IN` wählt nur Datensätze aus, deren Wert keinem der angegebenen Werte entspricht.

> **Merke:** `IN` = einer davon, `NOT IN` = keiner davon.

---

## 24. `IS NULL`

**Wann?** Wenn ein Wert fehlt.

```sql
WHERE t.ticketid IS NULL
```

**Begründung:**  
`IS NULL` wählt Datensätze aus, bei denen für die betreffende Spalte kein Wert vorhanden ist.

### ⭐ Wichtiger Fall: `LEFT JOIN` + `IS NULL`

**Aufgabe:** Kunden finden, die noch nie ein Ticket gekauft haben.

```sql
FROM customers c
LEFT JOIN tickets t
    ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
```

**Begründung:**  
Der `LEFT JOIN` behält alle Kunden. Existiert für einen Kunden kein Ticket, ist `t.ticketid` `NULL`; `WHERE t.ticketid IS NULL` wählt genau diese Kunden aus.

> **Merke:** „X, die kein Y haben“ → `LEFT JOIN Y` + `WHERE Y.id IS NULL`.

---

## 25. `IS NOT NULL`

**Wann?** Wenn nur Datensätze mit vorhandenem Wert benötigt werden.

```sql
WHERE customerid IS NOT NULL
```

**Begründung:**  
`IS NOT NULL` wählt nur Datensätze aus, bei denen für die betreffende Spalte ein Wert vorhanden ist.

---

# 🔢 Datentypen & Rundung

## 26. `ROUND`

**Wann?** Wenn ein numerisches Ergebnis gerundet werden soll.

```sql
ROUND(value, 2)
```

**Begründung:**  
`ROUND(..., 2)` rundet das Ergebnis auf zwei Dezimalstellen.

---

## 27. `CAST`

**Wann?** Wenn ein Wert in einen anderen Datentyp konvertiert werden soll.

```sql
CAST(value AS NUMERIC)
```

**Begründung:**  
`CAST(... AS NUMERIC)` konvertiert den Wert in den Datentyp `NUMERIC`.

**Beispiel:**

```sql
ROUND(CAST(AVG(t.paid_price) AS NUMERIC), 2)
```

**Begründung:**  
`AVG` berechnet den Durchschnitt, `CAST(... AS NUMERIC)` konvertiert das Ergebnis in `NUMERIC` und `ROUND(..., 2)` rundet es auf zwei Dezimalstellen.

---

## 28. `NUMERIC`

**Wann?** Für numerische Werte, insbesondere wenn Dezimalstellen benötigt werden.

```sql
CAST(value AS NUMERIC)
```

**Begründung:**  
`NUMERIC` wird als numerischer Datentyp verwendet und ermöglicht hier die gewünschte Rundung auf Dezimalstellen.

---

## 29. `INTEGER`

**Wann?** Wenn ein Wert als Ganzzahl benötigt wird.

```sql
CAST(value AS INTEGER)
```

**Begründung:**  
`INTEGER` repräsentiert ganze Zahlen ohne Nachkommastellen; `CAST(... AS INTEGER)` konvertiert den Wert entsprechend in eine Ganzzahl.

---

# ⭐ Wichtige Kombinationen für die Prüfung

## 1. „Alle X, auch ohne Y“

```sql
FROM X
LEFT JOIN Y ON ...
```

**Begründung:**  
`LEFT JOIN` wird verwendet, damit alle Datensätze aus `X` erhalten bleiben, auch wenn kein entsprechender Datensatz in `Y` existiert.

---

## 2. „X, die kein Y haben“

```sql
FROM X
LEFT JOIN Y ON ...
WHERE Y.id IS NULL
```

**Begründung:**  
Der `LEFT JOIN` behält alle Datensätze aus `X`. Existiert kein passender Datensatz in `Y`, ist `Y.id` `NULL`; `IS NULL` wählt genau diese Fälle aus.

---

## 3. „Anzahl Y pro X – inklusive X mit 0“

```sql
SELECT X.name, COUNT(Y.id)
FROM X
LEFT JOIN Y ON ...
GROUP BY X.name;
```

**Begründung:**  
Der `LEFT JOIN` sorgt dafür, dass alle `X` erhalten bleiben. `COUNT(Y.id)` ignoriert `NULL`-Werte und liefert deshalb für `X` ohne zugehöriges `Y` korrekt `0`.

---

## 4. „Durchschnitt / Anzahl / Summe pro Kategorie“

```sql
SELECT category, AVG(value)
FROM ...
GROUP BY category;
```

**Begründung:**  
`GROUP BY` fasst die Datensätze nach der Kategorie zusammen, sodass `AVG` den Durchschnitt für jede Gruppe separat berechnen kann.

---

## 5. „Nur Gruppen mit mehr als X“

```sql
GROUP BY category
HAVING COUNT(id) > X
```

**Begründung:**  
`GROUP BY` bildet die einzelnen Gruppen und `HAVING` filtert diese anhand des aggregierten `COUNT`-Werts.

---

## 6. Informationen liegen in zwei Tabellen

```sql
FROM table_a a
INNER JOIN table_b b
    ON a.id = b.id
```

**Begründungsvorlage:**  
`[Information A]` liegt in `[Tabelle A]`, während `[Information B]` in `[Tabelle B]` gespeichert ist. Deshalb werden beide Tabellen über `[gemeinsame ID]` verbunden.

---

## 7. Zwischentabelle notwendig

Beispiel:

`cinemas → screenings → tickets`

**Begründungsvorlage:**  
`[Tabelle A]` ist nicht direkt mit `[Tabelle C]` verbunden. Die Zuordnung erfolgt über `[Tabelle B]`, weshalb die drei Tabellen miteinander verbunden werden.

**Kino-Beispiel:**  
`tickets` ist nicht direkt mit `cinemas` verbunden. Ein Ticket gehört über `screeningid` zu einer Vorstellung und diese über `cinemaid` zu einem Kino, weshalb `tickets`, `screenings` und `cinemas` verbunden werden.

---

# 🚨 Schnellentscheidung an der Prüfung

| Aufgabenstellung enthält ... | Denken an ... |
|---|---|
| „gib diese Spalten aus“ | `SELECT` |
| „ohne Duplikate / unterschiedliche Werte“ | `DISTINCT` |
| „nur Datensätze, bei denen ...“ | `WHERE` |
| „beide Bedingungen“ | `AND` |
| „beginnt mit / enthält“ | `LIKE` |
| „pro Kino / pro Typ / pro Kunde“ | `GROUP BY` |
| „Gruppen mit mehr/weniger als ...“ | `HAVING` |
| „absteigend / aufsteigend“ | `ORDER BY` |
| „nur wenn auf beiden Seiten vorhanden“ | `INNER JOIN` |
| „alle X, auch ohne Y“ | `LEFT JOIN` |
| „X, die kein Y haben“ | `LEFT JOIN` + `IS NULL` |
| „über welche ID verbinden?“ | `ON` |
| „erste / höchste 5“ | `LIMIT` |
| „Durchschnitt“ | `AVG` |
| „Anzahl“ | `COUNT` |
| „kleinster Wert“ | `MIN` |
| „grösster Wert“ | `MAX` |
| „Gesamtsumme“ | `SUM` |
| „zwischen X und Y“ | `BETWEEN` |
| „einer dieser Werte“ | `IN` |
| „alle ausser diesen Werten“ | `NOT IN` |
| „fehlender Wert“ | `IS NULL` |
| „Wert muss vorhanden sein“ | `IS NOT NULL` |
| „2 Dezimalstellen“ | `ROUND(..., 2)` |
| „Datentyp umwandeln“ | `CAST` |

---

# 🧠 Die 5 goldenen Regeln

### 1. Alle X, auch ohne Y
→ `X LEFT JOIN Y`

### 2. X, die kein Y haben
→ `X LEFT JOIN Y` + `WHERE Y.id IS NULL`

### 3. Anzahl / Durchschnitt / Summe pro X
→ `GROUP BY X` + `COUNT / AVG / SUM`

### 4. Filter auf `COUNT / AVG / SUM`
→ `HAVING`, nicht `WHERE`

### 5. Informationen aus mehreren Tabellen
→ Überlegen: **Wo liegt welche Information und über welche IDs sind die Tabellen verbunden?**

---

# 📝 Universalvorlage für Begründungen

Bei einer Aufgabe zuerst drei Fragen beantworten:

1. **Wo liegen die benötigten Informationen?**
2. **Wie müssen die Tabellen verbunden werden?**
3. **Warum brauche ich Filter, Gruppierung oder Aggregation?**

Dann folgende Vorlage verwenden:

> `[Information A]` liegt in `[Tabelle A]` und `[Information B]` in `[Tabelle B]`, weshalb die Tabellen über `[ID]` verbunden werden. `[WHERE / GROUP BY / HAVING / Aggregatfunktion]` wird verwendet, um `[gewünschtes Ergebnis]` zu erhalten.

**Beispiel:**

> `membership_type` liegt in `customers` und `paid_price` in `tickets`, weshalb beide Tabellen über `customerid` verbunden werden. `GROUP BY membership_type` gruppiert die Tickets nach Mitgliedschaftstyp und `AVG(paid_price)` berechnet den durchschnittlich bezahlten Preis pro Gruppe.

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [70]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-03 10:38:17
Python Version: 3.14.2
IP Address: 127.0.0.1
-----------------------------------
